# Assignment 4, Problem 2: Mixed-boundary diffusion with FEniCSx

ME/BME 736 — Advanced Finite Element Methods (40 marks)

**Before you start.** Save a copy of this notebook as `lastname_assignment04_2d.ipynb` and select the kernel **Python (Finite Elements)**. Before you submit, run **Restart Kernel and Run All Cells** and check that every cell runs without an error.

Complete every cell marked **TODO**. You may add cells.

**Rules.** Use FEniCSx with continuous $P_1$ triangular elements. Start from the lecture notebook `notebooks/diffusion/diffusion_2d_demo.ipynb`, which solves a similar problem with different data. Copy code from it as needed, but do not edit that file. Chapter 6 covers the theory.

## The problem

Let $\Omega=(0,1)^2$. Prescribe $u$ on the vertical sides and the normal flux on the horizontal sides:

$$
-\Delta u=f\ \text{in }\Omega,
\qquad
u=g\ \text{on }\Gamma_D=\{x=0\}\cup\{x=1\},
\qquad
\boldsymbol q=-\nabla u,\qquad q=\boldsymbol q\cdot\boldsymbol n\ \text{on }\Gamma_N=\{y=0\}\cup\{y=1\}.
$$

The exact solution and the matching data are

$$
u_\star=1+x+x(1-x)(y+y^2),
\qquad
f=2(y+y^2)-2x(1-x),
\qquad
g=1+x,
$$

$$
q=x(1-x)\ \text{on }y=0,
\qquad
q=-3x(1-x)\ \text{on }y=1.
$$

The flux differs on the bottom and top sides, so each side needs its own facet tag and its own boundary integral in the linear functional. With the outward-flux convention, $\ell(v)=\int_\Omega fv\,\mathrm{d}x-\int_{\Gamma_N}qv\,\mathrm{d}s$.

In [ ]:
from mpi4py import MPI                      # parallel communicator; we use MPI.COMM_SELF (one process)
from petsc4py import PETSc                  # linear algebra library used by FEniCSx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri               # plotting on triangle meshes
import ufl                                  # language for writing variational forms
from dolfinx import fem, mesh, plot         # FEniCSx: function spaces, meshes, plotting data
import dolfinx.fem.petsc
from dolfinx.fem.petsc import LinearProblem # assembles and solves a(u, v) = l(v)

MESHES = [4, 8, 16, 32]  # n for the n x n meshes

## Plotting helpers

Run this cell once. These are the helpers from the lecture notebook. `draw_field(ax, V, values, title)` plots nodal values on the mesh.

In [ ]:
def triangulation_for(V):
    cells, _, coordinates = plot.vtk_mesh(V)
    triangles = cells.reshape(-1, 4)[:, 1:]
    tri = mtri.Triangulation(coordinates[:, 0], coordinates[:, 1], triangles)
    return tri, coordinates, triangles


def draw_field(ax, V, values, title, show_mesh=True):
    tri, _, _ = triangulation_for(V)
    image = ax.tripcolor(tri, values, shading="gouraud", cmap="viridis")
    if show_mesh:
        ax.triplot(tri, color="black", linewidth=0.7, alpha=0.65)
    ax.set(xlabel="$x$", ylabel="$y$", title=title, aspect="equal")
    return image

## Step 1. Solve on one mesh (TODO)

Write `solve_diffusion(n)` for a uniform $n\times n$ subdivision of the square with two triangles per square:
`mesh.create_unit_square(MPI.COMM_SELF, n, n, cell_type=mesh.CellType.triangle)`. It should:

1. build the $P_1$ space and write $u_\star$, $f$, and the two values of $q$ in UFL;
2. find the left and right facets and impose $u=g$ there;
3. tag the bottom facets 1 and the top facets 2, and use `ds(1)` and `ds(2)` in the linear functional;
4. solve; and
5. compute three norms of the error $e=u_\star-u_h$, each with a quadrature degree of at least 6:

$$
\|e\|_{L^2(\Omega)}=\left(\int_\Omega e^2\,\mathrm{d}x\right)^{1/2},
\qquad
\|e\|_{H^1(\Omega)}=\left(\int_\Omega \left(e^2+\nabla e\cdot\nabla e\right)\mathrm{d}x\right)^{1/2},
\qquad
\|e\|_{E}=\sqrt{a(e,e)}=\left(\int_\Omega \nabla e\cdot\nabla e\,\mathrm{d}x\right)^{1/2}.
$$

The lecture notebook computes the $L^2$ error. Extend the same lines to the other two. For the energy norm, evaluate the bilinear form $a(u,v)=\int_\Omega\nabla u\cdot\nabla v\,\mathrm{d}x$ with $u=v=e$, then take the square root.

Return what the later steps need, for example the mesh, the space, $u_h$, and the three errors.

In [ ]:
def solve_diffusion(n):
    # TODO
    raise NotImplementedError

## Step 2. Mesh refinement (TODO)

Solve for $n=4$, $8$, $16$, and $32$, and use $h=\sqrt2/n$ as the mesh size.

1. Print a table of $n$, the number of triangles, the number of degrees of freedom, $h$, and the three errors.
2. On one figure with logarithmic axes, plot the three errors against $h$. Add two reference lines, one with slope 1 (proportional to $h$) and one with slope 2 (proportional to $h^2$).
3. Print the observed rate of each error between consecutive meshes. The formula for rate using mesh size for $n$ and $2n$ is $\log(e_n/e_{2n})/\log 2$.

In [ ]:
# TODO

## Step 3. Solution on the $8\times8$ mesh (TODO)

Plot $u_h$ on the $8\times8$ mesh with `draw_field`, as in section 3 of the lecture notebook. The mesh lines are drawn on top of the solution.

In [ ]:
# TODO

## Step 4. Check the prescribed values (TODO)

Print the largest difference between $u_h$ and $g=1+x$ at the degrees of freedom on $x=0$ and $x=1$. Hint: `fem.locate_dofs_topological` gives these degrees of freedom, as in the lecture notebook, and `V.tabulate_dof_coordinates()` gives their coordinates.

In [ ]:
# TODO

## Conclusions (TODO)

In a few sentences, state:

1. whether the prescribed values hold on both vertical sides; and
2. the observed rates. Compare the $L^2$ error with the slope-2 line, and the $H^1$ and energy errors with the slope-1 line.